
## 1. Data preprocessing  

### === 1_1_ Reading the data from World Bank's API =======

In [ ]:
! pip install pandas_datareader

In [1]:
from pandas_datareader import wb

Go to 

https://databank.worldbank.org/source/world-development-indicators

In [2]:
code = 'EN.CO2.BLDG.ZS'

CO2 emissions from residential buildings and from commercial and from public services (% of total fuel combustion)

Countries:

In [ ]:
regions = ['BRA',  'IND', 'CHN', 'ZAF', 'USA', 'GBR', 'WLD', 'EUU']

BRA: Brazil

IND: India

CHN: China
    
ZAF: South Africa
    
USA: United States
    
GBR: UK
    
WLD: World
    
EUU: European Union


Another interesting indicator:  'EG.ELC.ACCS.ZS'
electricity access


In [5]:
import pandas as pd

try:
    df_indicator = wb.download(
        country=regions,
        indicator=code,
        start=1971, 
        end=2014
    )
except: 
    df_indicator = pd.read_excel("indicator_world_bank_original.xlsx", index_col = [0,1])
    # If the connection to worldbank does not work, then you will load the data from this file. 
    # make sure you have this excel file i.e. indicator_world_bank_original.xlsx in the python working directory.
    # you can find this file if you search in the data in this course.
    # Also: even if the connection to worldbank works, you can still use this excel file if you want to 
    # use the exact same data as the ones that are in the videos of this course. Because otherwise, 
    # worldbank's data may change from time to time. Eg WorldBank may add more data or change some values of 
    # existing data.
    

df_indicator

EN.CO2.BLDG.ZS
country      year                
Brazil       2014        4.289736
             2013        4.508896
             2012        4.746116
             2011        5.159270
             2010        5.346396
...                           ...
South Africa 1975        6.572400
             1974        6.815974
             1973        6.949115
             1972        7.742371
             1971        7.925393

[352 rows x 1 columns]

The data contains the CO2 emission from buildings (including residential, commercial and public services) as a percentage of fuel consumption. 

This is a times-series dataset because each date has a value. Here every year has a single value.

In [ ]:
df_indicator.to_excel("indicator_world_bank.xlsx")

these are small datasets and ideally for ML we would like to have hundreds of datapoints. 

##### allowing all rows to be seen

In [ ]:
import pandas as pd
pd.set_option("display.max_rows", None)

In [ ]:
df_indicator.head(9)

##### checking for NaN values

In [ ]:
df_indicator.isnull().sum()

### ================= 1_2_ Converting the elements of column Year , from strings into integers ===========

In [ ]:
df_indicator_backup1 = df_indicator.copy()  # for later use 

In [ ]:
df_indicator = df_indicator.reset_index()


In [ ]:
df_indicator.head(3)

In [ ]:
type(df_indicator['year'][0])

In [ ]:
df_indicator['year'] = df_indicator['year'].astype(int)
type(df_indicator['year'][0])

The column year, initially was a string. Its values were string. Is this problematic?

Yes, because we use the year to construct the time index. The t.

And t , we then use it as a feature (after taking the polynomials), so it must be numeric.

And in this case we use the year as a feature, so its type must be either int or float, not string or datetime.
So making the year datetime would not be correct!

Later in the code we apply the polynomial transformer to `df_indicator.index.values`, which doesn't work if the index is string or datetime.

### ======= 1_3_ Setting "year" to be the index again (now its elements being integers) ===============

In [ ]:
df_indicator

In [ ]:
df_indicator.set_index('year', inplace= True)

df_indicator.head()

### ======================== 1_4_ Sorting the index (so that 1971 is on top)======================

In [ ]:
df_indicator

In [ ]:
df_indicator = df_indicator.sort_index()
df_indicator

Above we sort them so that 1971 is on the top. Is there any difference if 1971 or 2014 is on the top ? Or just for visual purposes ie we like having on the top the 1971? 

The code will still work, but the results won't make any sense. 
The ARIMA model assumes that `y(t)` is preceded by `y(t - 1)` in the input time series, but that won't be the case if the time index is sorted in descending order instead of ascending.

In [ ]:
df_indicator.columns = ["country" ,"Indicator"]
df_indicator.head()

### ======================= 1_5_ Columns are countries=====================================

In [ ]:
df_indicator.head(10)

In [ ]:
df_indicator = pd.pivot_table(df_indicator, values='Indicator', 
                    columns='country', index='year')


In [ ]:
df_indicator



In [ ]:
regions # we want these codes to be the columns of df_indicator. 
#This will facilitate for-loops later on.
#We dont care if the order of columns in df_indicator is same as 
#in regions

In [ ]:
df_indicator.columns

In [ ]:
# PAY ATTENTION HERE SO THAT YOU PUT THEM IN THE RIGHT ORDER ! 

df_indicator.columns = [regions[0], regions[2], regions[7], regions[1], regions[3], regions[5], regions[4], regions[6] ]

df_indicator

In [ ]:
df_indicator.tail(10)

In [ ]:
df_indicator.to_excel('dfinc.xlsx')